# Novel Tail Split Investigation

This notebook investigates the proposed novelty-focused validation split without wiring it into the validation pipeline.

It now also supports a post-split filter:
- keep only held-out targets whose item still has at least `MIN_TRAINING_CUSTOMER_SUPPORT` customers in the truncated training set

That lets you answer the main concern directly: are we validating on items with too little remaining support?

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from validation.offline import hold_out_last_novel_tail_item

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

DATA_PATH = Path(r'C:\Users\lovro\Desktop\hackatoni\LUMEN_DS_processed.csv')
USER_COL = 'CustomerID'
ITEM_COL = 'item_idx'
DATE_COL = 'Order Date'
TAIL_FRACTION = 0.2
MIN_TRAINING_CUSTOMER_SUPPORT = 3

df = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df):,} purchases from {DATA_PATH}')
df[[USER_COL, 'Item Code', ITEM_COL, DATE_COL]].head()

In [ ]:
split = hold_out_last_novel_tail_item(
    df,
    user_col=USER_COL,
    item_col=ITEM_COL,
    date_col=DATE_COL,
    tail_fraction=TAIL_FRACTION,
)

masked_df = split.masked_df
eligible_df = pd.DataFrame(split.split_details['eligible_records'])
heldout_targets = pd.Series(split.ground_truth, name='heldout_item_idx')
heldout_targets.index.name = USER_COL

original_support = df.groupby(ITEM_COL)[USER_COL].nunique().rename('original_customer_support')
training_support = masked_df.groupby(ITEM_COL)[USER_COL].nunique().rename('training_customer_support')
heldout_count = heldout_targets.value_counts().rename('heldout_customer_count')

item_meta = (
    df[[ITEM_COL, 'Item Code', 'Product family', 'Product group']]
    .drop_duplicates(ITEM_COL)
    .set_index(ITEM_COL)
)

item_support = item_meta.join(original_support, how='left').join(training_support, how='left').join(heldout_count, how='inner')
item_support.index.name = ITEM_COL
item_support['original_customer_support'] = item_support['original_customer_support'].fillna(0).astype(int)
item_support['training_customer_support'] = item_support['training_customer_support'].fillna(0).astype(int)
item_support['heldout_customer_count'] = item_support['heldout_customer_count'].fillna(0).astype(int)
item_support['support_loss'] = item_support['original_customer_support'] - item_support['training_customer_support']
item_support['heldout_fraction_of_original_support'] = item_support['heldout_customer_count'] / item_support['original_customer_support'].replace(0, pd.NA)
item_support = item_support.reset_index()

base_summary = pd.Series(
    {
        'tail_fraction': TAIL_FRACTION,
        'min_training_customer_support': MIN_TRAINING_CUSTOMER_SUPPORT,
        'validation_customers_before_support_filter': len(split.evaluated_user_ids),
        'unique_heldout_items_before_support_filter': int(heldout_targets.nunique()),
        'original_purchases': len(df),
        'training_purchases_after_temporal_split': len(masked_df),
        'purchases_removed_by_temporal_split': len(df) - len(masked_df),
    }
)
base_summary

In [ ]:
eligible_item_ids = set(
    item_support.loc[
        item_support['training_customer_support'] >= MIN_TRAINING_CUSTOMER_SUPPORT,
        ITEM_COL,
    ].tolist()
)

filtered_eligible_df = eligible_df[eligible_df['heldout_item'].isin(eligible_item_ids)].copy()
filtered_ground_truth = {
    user_id: item_id
    for user_id, item_id in split.ground_truth.items()
    if item_id in eligible_item_ids
}

filtered_item_support = item_support[item_support[ITEM_COL].isin(eligible_item_ids)].copy()

filtered_summary = pd.Series(
    {
        'validation_customers_after_support_filter': len(filtered_ground_truth),
        'customers_removed_by_support_filter': len(split.evaluated_user_ids) - len(filtered_ground_truth),
        'unique_heldout_items_after_support_filter': int(filtered_item_support[ITEM_COL].nunique()),
        'heldout_items_removed_by_support_filter': int(item_support[ITEM_COL].nunique() - filtered_item_support[ITEM_COL].nunique()),
    }
)
filtered_summary

In [ ]:
support_threshold_table = []
for threshold in [0, 1, 2, 3, 5, 10, 20]:
    kept_item_ids = set(item_support.loc[item_support['training_customer_support'] >= threshold, ITEM_COL].tolist())
    kept_customers = int(eligible_df['heldout_item'].isin(kept_item_ids).sum())
    kept_items = int(item_support.loc[item_support['training_customer_support'] >= threshold, ITEM_COL].nunique())
    support_threshold_table.append(
        {
            'min_training_customer_support': threshold,
            'kept_validation_customers': kept_customers,
            'dropped_validation_customers': len(split.evaluated_user_ids) - kept_customers,
            'kept_heldout_items': kept_items,
            'dropped_heldout_items': int(item_support[ITEM_COL].nunique() - kept_items),
        }
    )

pd.DataFrame(support_threshold_table)

In [ ]:
lowest_support_items_before_filter = item_support.sort_values(
    ['training_customer_support', 'heldout_customer_count', 'original_customer_support', ITEM_COL],
    ascending=[True, False, True, True],
)
lowest_support_items_before_filter.head(25)

In [ ]:
lowest_support_items_after_filter = filtered_item_support.sort_values(
    ['training_customer_support', 'heldout_customer_count', 'original_customer_support', ITEM_COL],
    ascending=[True, False, True, True],
)
lowest_support_items_after_filter.head(25)

In [ ]:
repeated_and_weak_before_filter = item_support[
    (item_support['heldout_customer_count'] >= 2) & (item_support['training_customer_support'] <= 5)
].sort_values(['heldout_customer_count', 'training_customer_support'], ascending=[False, True])
repeated_and_weak_before_filter

In [ ]:
repeated_items_after_filter = filtered_item_support[
    filtered_item_support['heldout_customer_count'] >= 2
].sort_values(['heldout_customer_count', 'training_customer_support'], ascending=[False, True])
repeated_items_after_filter.head(25)

In [ ]:
weak_support_before_filter = item_support[item_support['training_customer_support'] <= MIN_TRAINING_CUSTOMER_SUPPORT].copy()

family_summary_before_filter = (
    weak_support_before_filter.groupby('Product family')
    .agg(
        heldout_items=('Item Code', 'count'),
        total_heldout_customers=('heldout_customer_count', 'sum'),
        mean_training_support=('training_customer_support', 'mean'),
    )
    .sort_values(['heldout_items', 'total_heldout_customers'], ascending=[False, False])
)

group_summary_before_filter = (
    weak_support_before_filter.groupby('Product group')
    .agg(
        heldout_items=('Item Code', 'count'),
        total_heldout_customers=('heldout_customer_count', 'sum'),
        mean_training_support=('training_customer_support', 'mean'),
    )
    .sort_values(['heldout_items', 'total_heldout_customers'], ascending=[False, False])
)

family_summary_before_filter.head(15), group_summary_before_filter.head(15)